# Tourism demand forecasting starter

## Goal

Build a cutoff-safe, reproducible workflow for point and interval tourism-demand forecasting. The scaffold supplies data loading, temporal partitions, weak point baselines, metric helpers, output contracts, and safe export checks. Your group must design and justify the improved point model and probabilistic interval method.

This notebook contains no hidden actuals, competitive forecasts, tuned parameters, or private scoring bands.

## Setup

Run the notebook from the released assignment folder or upload the whole folder to an online environment. The loader uses `TOURISM_DATA_PATH` when set, then a local `ISF-TDF2023.csv`, and otherwise the approved public raw URL.

Complete `GROUP_ID` before final export. Keep all preprocessing, tuning, interval calibration, and external data cutoff-safe.

In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

GROUP_ID = 'Group-NAME'  # Replace with the assigned identifier.
DATA_URL = (
    'https://raw.githubusercontent.com/tulip-lab/open-data/main/'
    'ISF-TDF2023/ISF-TDF2023.csv'
)
PUBLIC_CUTOFF = '2023M07'
TRAIN_END = '2022M07'
VALIDATION_MONTHS = tuple(
    [f'2022M{month:02d}' for month in range(8, 13)]
    + [f'2023M{month:02d}' for month in range(1, 3)]
)
AUDIT_MONTHS = tuple(f'2023M{month:02d}' for month in range(3, 8))
FORECAST_MONTHS = tuple(
    [f'2023M{month:02d}' for month in range(8, 13)]
    + [f'2024M{month:02d}' for month in range(1, 8)]
)
INTERVAL_COLUMNS = (
    'Date', 'destination', 'point', 'lower80', 'upper80',
    'lower95', 'upper95', 'model_label',
)
RANDOM_SEED = 742
PROTOCOL_FROZEN = False  # Change to True only after completing Protocol Sections A–C.
np.random.seed(RANDOM_SEED)
print({
    'group_id': GROUP_ID, 'seed': RANDOM_SEED,
    'protocol_frozen': PROTOCOL_FROZEN,
})

## Data

### 1. Load and validate the public source

Dataset: ISF-TDF2023, TULIP Lab Open Data Repository, CC BY 4.0 unless otherwise stated. The assessment-period rows are intentionally blank.

In [ ]:
configured_path = os.environ.get('TOURISM_DATA_PATH', '').strip()
candidate_paths = [Path(configured_path)] if configured_path else []
candidate_paths.append(Path('ISF-TDF2023.csv'))
local_path = next((path for path in candidate_paths if path.is_file()), None)
data_source = local_path if local_path is not None else DATA_URL
raw_tourism_data = pd.read_csv(data_source)

assert raw_tourism_data.columns[0] == 'Date'
assert raw_tourism_data['Date'].is_unique
destinations = tuple(raw_tourism_data.columns[1:])
assert len(destinations) == 20, f'Expected 20 destinations, found {len(destinations)}'
required_months = (*VALIDATION_MONTHS, *AUDIT_MONTHS, *FORECAST_MONTHS)
assert set(required_months).issubset(set(raw_tourism_data['Date']))
final_cells = raw_tourism_data.loc[
    raw_tourism_data['Date'].isin(FORECAST_MONTHS), list(destinations)
]
assert final_cells.isna().all().all(), 'Final-period targets must be blank.'
print({
    'data_source': str(data_source),
    'rows': len(raw_tourism_data),
    'destinations': len(destinations),
    'public_cutoff': PUBLIC_CUTOFF,
})

### 2. Create the fixed temporal partitions

Do not use the locked audit for model, feature, hyperparameter, or interval-width selection. Open it once only after completing the Protocol. Missing history is not zero; document any cutoff-safe imputation.

In [ ]:
historical_actual_wide = raw_tourism_data.loc[
    raw_tourism_data['Date'] <= PUBLIC_CUTOFF, ['Date', *destinations]
].copy()
training_actual_wide = historical_actual_wide.loc[
    historical_actual_wide['Date'] <= TRAIN_END
].copy()
validation_actual_wide = historical_actual_wide.loc[
    historical_actual_wide['Date'].isin(VALIDATION_MONTHS)
].copy()
missing_history = historical_actual_wide[list(destinations)].isna().sum()
partition_summary = pd.DataFrame({
    'partition': ['training', 'validation', 'locked_audit', 'final_forecast'],
    'months': [
        len(training_actual_wide), len(validation_actual_wide),
        len(AUDIT_MONTHS), len(FORECAST_MONTHS),
    ],
    'start': [
        training_actual_wide['Date'].iloc[0], VALIDATION_MONTHS[0],
        AUDIT_MONTHS[0], FORECAST_MONTHS[0],
    ],
    'end': [TRAIN_END, VALIDATION_MONTHS[-1], AUDIT_MONTHS[-1], FORECAST_MONTHS[-1]],
})
print('Public-history missing values:', int(missing_history.sum()))
partition_summary

## Shared evaluation helpers

These helpers standardise routine evaluation so groups can concentrate on model design. MASE uses lag-1 scaling from the supplied training history. MAPE excludes zero actuals. WIS combines point error with the submitted 80% and 95% intervals.

In [ ]:
def shift_month(label: str, offset: int) -> str:
    year, month = int(label[:4]), int(label[-2:])
    ordinal = year * 12 + month - 1 + offset
    return f'{ordinal // 12}M{ordinal % 12 + 1:02d}'

def recursive_naive_forecast(
    history_wide: pd.DataFrame, cutoff: str, forecast_months: tuple[str, ...], lag: int
) -> pd.DataFrame:
    if lag <= 0:
        raise ValueError('lag must be a positive integer')
    available = history_wide.loc[history_wide['Date'] <= cutoff, ['Date', *destinations]]
    values = {
        row['Date']: pd.to_numeric(row[list(destinations)], errors='coerce').to_numpy(float)
        for _, row in available.iterrows()
    }
    output_rows = []
    for month in forecast_months:
        source_month = shift_month(month, -lag)
        if source_month not in values:
            raise ValueError(f'No lag-{lag} source for {month}: {source_month}')
        values[month] = values[source_month].copy()
        output_rows.append({
            'Date': month, **dict(zip(destinations, values[month], strict=True))
        })
    return pd.DataFrame(output_rows, columns=['Date', *destinations])

def evaluate_point_forecast(
    forecast_wide: pd.DataFrame, actual_wide: pd.DataFrame, scale_history_wide: pd.DataFrame
) -> pd.DataFrame:
    forecast = forecast_wide.set_index('Date')
    actual = actual_wide.set_index('Date')
    rows = []
    for destination in destinations:
        paired = pd.concat(
            [actual[destination].rename('actual'), forecast[destination].rename('forecast')],
            axis=1, join='inner',
        ).apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        errors = (paired['actual'] - paired['forecast']).abs()
        training = pd.to_numeric(
            scale_history_wide[destination], errors='coerce'
        ).replace([np.inf, -np.inf], np.nan).dropna()
        denominator = training.diff().abs().dropna().mean()
        nonzero = paired['actual'] != 0
        rows.append({
            'destination': destination, 'n': len(paired), 'mae': errors.mean(),
            'mase': errors.mean() / denominator if denominator > 0 else np.nan,
            'mape': (errors[nonzero] / paired.loc[nonzero, 'actual'].abs()).mean() * 100
                if nonzero.any() else np.nan,
            'mase_denominator': denominator, 'mape_valid_n': int(nonzero.sum()),
        })
    return pd.DataFrame(rows)

def interval_score(actual: float, lower: float, upper: float, alpha: float) -> float:
    score = upper - lower
    if actual < lower:
        score += (2 / alpha) * (lower - actual)
    elif actual > upper:
        score += (2 / alpha) * (actual - upper)
    return score

def evaluate_interval_forecast(
    intervals_long: pd.DataFrame, actual_wide: pd.DataFrame
) -> tuple[pd.DataFrame, dict[str, float]]:
    if set(intervals_long.columns) != set(INTERVAL_COLUMNS):
        raise ValueError(f'Interval columns must be exactly {list(INTERVAL_COLUMNS)}')
    actual_long = actual_wide.melt(
        id_vars='Date', value_vars=list(destinations),
        var_name='destination', value_name='actual',
    )
    scored = intervals_long.merge(actual_long, on=['Date', 'destination'], how='inner')
    numeric = ['point', 'lower80', 'upper80', 'lower95', 'upper95', 'actual']
    scored[numeric] = scored[numeric].apply(pd.to_numeric, errors='coerce')
    if scored[numeric].isna().any().any() or not np.isfinite(scored[numeric]).all().all():
        raise ValueError('Intervals and actuals must be finite numeric values')
    nested = (
        (scored['lower95'] <= scored['lower80'])
        & (scored['lower80'] <= scored['point'])
        & (scored['point'] <= scored['upper80'])
        & (scored['upper80'] <= scored['upper95'])
    )
    if not nested.all():
        raise ValueError('All intervals must be nested around the point forecast')
    scored['covered80'] = scored['actual'].between(scored['lower80'], scored['upper80'])
    scored['covered95'] = scored['actual'].between(scored['lower95'], scored['upper95'])
    scored['width80'] = scored['upper80'] - scored['lower80']
    scored['width95'] = scored['upper95'] - scored['lower95']
    scored['wis'] = scored.apply(
        lambda row: (
            0.5 * abs(row.actual - row.point)
            + 0.10 * interval_score(row.actual, row.lower80, row.upper80, 0.20)
            + 0.025 * interval_score(row.actual, row.lower95, row.upper95, 0.05)
        ) / 0.625, axis=1,
    )
    by_destination = scored.groupby('destination', as_index=False).agg(
        n=('actual', 'size'), mean_wis=('wis', 'mean'),
        picp80=('covered80', 'mean'), picp95=('covered95', 'mean'),
        mpiw80=('width80', 'mean'), mpiw95=('width95', 'mean'),
    )
    aggregate = {
        'n': int(len(scored)), 'mean_wis': float(scored['wis'].mean()),
        'picp80': float(scored['covered80'].mean()),
        'picp95': float(scored['covered95'].mean()),
        'ace': float((abs(scored['covered80'].mean() - 0.80)
            + abs(scored['covered95'].mean() - 0.95)) / 2),
    }
    return by_destination, aggregate

## Steps

### 3. Establish the required weak point baselines

The scaffold implements lag-1 and lag-12 so every group starts from the same cutoff-safe comparison. Inspect and understand these results before developing an improved method.

In [ ]:
lag1_validation_forecast = recursive_naive_forecast(
    training_actual_wide, TRAIN_END, VALIDATION_MONTHS, lag=1
)
lag12_validation_forecast = recursive_naive_forecast(
    training_actual_wide, TRAIN_END, VALIDATION_MONTHS, lag=12
)
baseline_validation_metrics = pd.concat(
    [
        evaluate_point_forecast(
            lag1_validation_forecast, validation_actual_wide, training_actual_wide
        ).assign(model_label='naive_lag1'),
        evaluate_point_forecast(
            lag12_validation_forecast, validation_actual_wide, training_actual_wide
        ).assign(model_label='naive_lag12'),
    ], ignore_index=True,
)
baseline_validation_summary = baseline_validation_metrics.groupby(
    'model_label', as_index=False
).agg(
    n_destinations=('destination', 'nunique'),
    mean_mase=('mase', 'mean'), median_mase=('mase', 'median'),
    mean_mape=('mape', 'mean'),
)
baseline_validation_summary

### 4. Develop and validate an improved point method

This is core assessed work. Use the validation period for features, preprocessing, hyperparameters, and selection. Compare the candidate with both baseline tables above. Preserve destination-specific and horizon-specific failures.

In [ ]:
# TODO: implement at least one improved point-forecast method.
# Expected wide output columns: Date followed by the 20 destination columns.
improved_validation_forecast = pd.DataFrame(columns=['Date', *destinations])
improved_validation_metrics = pd.DataFrame(
    columns=['destination', 'n', 'mae', 'mase', 'mape', 'mase_denominator', 'mape_valid_n']
)
point_model_summary = pd.DataFrame(columns=[
    'model_label', 'mean_mase', 'median_mase', 'mean_mape',
    'selected', 'selection_rationale', 'runtime', 'stability_notes',
])

### 5. Develop and assess predictive intervals

Compare a simple residual or seasonal-bootstrap interval baseline with at least one improved probabilistic method. The interval algorithm is core assessed work and is therefore not implemented here. Use the supplied evaluator helper to interpret coverage and sharpness/WIS together.

Use the paired fictional CSV examples only to inspect the required shape. Submitted bounds must be generated from the documented method; do not type or guess them.

In [ ]:
# TODO: generate validation intervals for the baseline and improved method.
baseline_validation_intervals = pd.DataFrame(columns=INTERVAL_COLUMNS)
improved_validation_intervals = pd.DataFrame(columns=INTERVAL_COLUMNS)
interval_model_summary = pd.DataFrame(columns=[
    'model_label', 'mean_wis', 'picp80', 'picp95', 'ace',
    'selected', 'selection_rationale', 'runtime', 'stability_notes',
])
# When populated, evaluate with:
# interval_by_destination, interval_aggregate = evaluate_interval_forecast(
#     improved_validation_intervals, validation_actual_wide
# )

### 6. Freeze the Protocol, then open the locked audit once

Complete Sections A–C of `templates/Forecast-Protocol.md` first. Run the frozen workflow on `AUDIT_MONTHS` once, retain mixed or negative results, and record the result in Section D. Later changes are exploratory and must not replace the original audit evidence.

In [ ]:
locked_audit_actual_wide = None
if PROTOCOL_FROZEN:
    locked_audit_actual_wide = historical_actual_wide.loc[
        historical_actual_wide['Date'].isin(AUDIT_MONTHS)
    ].copy()
    print('Audit unlocked for the frozen workflow:', list(AUDIT_MONTHS))
else:
    print('Audit remains locked. Complete Protocol Sections A–C first.')

# TODO: after unlocking, run the frozen point and interval workflow once.
locked_audit_point_results = pd.DataFrame()
locked_audit_interval_results = pd.DataFrame()
audit_notes = {
    'protocol_frozen_before_run': False,
    'run_identifier': '',
    'deviations': 'none',
}

### 7. Refit the selected workflow and create final tables

Refit through `2023M07` only. The point and interval tables below are the exact objects exported for scoring. Do not edit the exported CSV files manually.

In [ ]:
# TODO: replace these empty contracts with outputs from the selected final workflow.
forecast_submission_wide = pd.DataFrame(columns=['Date', *destinations])
interval_submission_long = pd.DataFrame(columns=INTERVAL_COLUMNS)

## Export and structural validation

The export helper enforces the published table contracts and exact group filenames. It does not measure hidden-period forecast quality.

In [ ]:
def export_submission(
    forecast_wide: pd.DataFrame, intervals_long: pd.DataFrame, group_id: str
) -> tuple[Path, Path]:
    if re.fullmatch(r'Group-[A-Za-z0-9]{1,8}', group_id) is None:
        raise ValueError('GROUP_ID must match Group-NAME with 1–8 letters or digits')
    if tuple(forecast_wide.columns) != ('Date', *destinations):
        raise ValueError('Forecast columns or column order are incorrect')
    if tuple(forecast_wide['Date']) != FORECAST_MONTHS:
        raise ValueError('Forecast rows must be the 12 required months in order')
    if tuple(intervals_long.columns) != INTERVAL_COLUMNS:
        raise ValueError('Interval columns or column order are incorrect')
    expected_keys = {
        (month, destination) for month in FORECAST_MONTHS for destination in destinations
    }
    actual_keys = set(zip(
        intervals_long['Date'], intervals_long['destination'], strict=True
    ))
    if actual_keys != expected_keys or len(intervals_long) != len(expected_keys):
        raise ValueError('Intervals must contain each required destination-month exactly once')
    forecast_path = Path(f'A2-{group_id}-Forecast.csv')
    interval_path = Path(f'A2-{group_id}-Intervals.csv')
    forecast_wide.to_csv(forecast_path, index=False)
    intervals_long.to_csv(interval_path, index=False)
    return forecast_path, interval_path

ready_to_export = (
    GROUP_ID != 'Group-NAME'
    and len(forecast_submission_wide) == len(FORECAST_MONTHS)
    and len(interval_submission_long) == len(FORECAST_MONTHS) * len(destinations)
)
if ready_to_export:
    forecast_path, interval_path = export_submission(
        forecast_submission_wide, interval_submission_long, GROUP_ID
    )
    print('Generated:', forecast_path, interval_path)
    print('Now run:')
    print(f'python validate_submission.py {forecast_path} {interval_path}')
else:
    print('Starter mode: complete GROUP_ID and both final tables before export.')

## Checks

Before submission:

1. restart the kernel and run all cells from top to bottom;
2. confirm that the notebook itself regenerates both exact group CSV files;
3. run `validate_submission.py` on those files and retain the PASS output;
4. complete the Protocol, Report, Model Card, Contribution Declaration, and Submission Checklist; and
5. extract the final ZIP in a new location and confirm every required file opens.

In [ ]:
assert len(FORECAST_MONTHS) == 12
assert len(destinations) == 20
assert len(FORECAST_MONTHS) * len(destinations) == 240
print('Expected final cases: 240')

## Next steps

Implement the improved point and interval algorithms, record the frozen selection rules before the audit, and keep conclusions tied to visible validation and audit evidence. Do not place hidden actuals, credentials, private links, or personal data in notebooks, prompts, services, caches, or output files.